# Пилот корпуса Википедии

Собираем статьи Википедии по 8 пилотным темам (`data/tasks/pilot_topics.yaml`) и сохраняем выдачу
живого поиска Википедии: по ней потом подбираются веса нашего BM25 и проверяется, находятся ли разделы
с понятиями из тезисов.

Запросы по теме: сама тема и ключевые слова тезисов (`terms`) трёх составителей (пилот 3) — короткие
запросы: длинные фразы поиск Википедии понимает плохо (первый запуск, по заголовкам тезисов, это показал).
Для каждой темы берём статьи из выдачи русской Википедии, статьи из ссылок во введениях трёх лучших статей
по теме, английские версии найденных статей по межъязыковым ссылкам и выдачу английской Википедии по
английским названиям трёх лучших статей.
Код — `src/presentation_rl/tools/wiki.py` и `tools/corpus.py`.

Нужен интернет, видеокарта не нужна. Около 1000–1500 запросов к API, примерно 10–15 минут
(Википедия просит не спешить: пауза 0,5 с между запросами). Сборку можно прервать и запустить снова —
скачанное повторно не запрашивается.

Википедия требует указать контакт в заголовке запросов: перед запуском задайте переменную окружения
`WIKI_CONTACT` (почта или ссылка на страницу проекта) или впишите контакт в `CONTACT` ниже.

Результат — папка `data/sources/wiki_pilot/` (в git не попадает): `articles.jsonl` (общий для запусков:
уже скачанные статьи повторно не качаются) и `topics_v2.jsonl` (`topics.jsonl` — первый запуск по заголовкам тезисов).

In [4]:
import json, os, sys
from pathlib import Path

import yaml

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl.tools.corpus import CorpusStore, TopicPlan, collect
from presentation_rl.tools.wiki import WikiClient

CONTACT = os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")  # или впишите сюда почту / ссылку на проект
OUT = ROOT / "data" / "sources" / "wiki_pilot"

In [5]:
# Темы и запросы: тема + ключевые слова тезисов (terms) всех трёх составителей без повторов.
topics = yaml.safe_load((ROOT / "data/tasks/pilot_topics.yaml").read_text(encoding="utf-8"))["topics"]
drafts = ["claude/pilot3.json", "codex/pilot3.json", "gemini/pilot3_notool.json"]
terms = {t["id"]: [] for t in topics}
for name in drafts:
    data = json.loads((ROOT / "data/tasks/drafts" / name).read_text(encoding="utf-8"))
    for t in data["topics"]:
        for point in t["key_points"]:
            for term in point["terms"]:
                if term.lower() not in (x.lower() for x in terms[t["id"]]):
                    terms[t["id"]].append(term)

plans = [TopicPlan(t["id"], t["topic"], terms[t["id"]]) for t in topics]
for p in plans:
    print(f"{p.id}: {len(p.queries)} запросов, например: {', '.join(p.queries[:5])}")

gradient_descent: 77 запросов, например: целевая функция, антиградиент, направление наискорейшего убывания, линейная аппроксимация, шаг обучения
photosynthesis: 71 запросов, например: оксигенный фотосинтез, окислительно-восстановительные реакции, донор электронов, автотрофы, тилакоиды
serf_reform: 81 запросов, например: поражение в Крымской войне, кризис крепостного хозяйства, крестьянские волнения, экономическое отставание, Секретный (Главный) комитет
entropy: 73 запросов, например: количество информации, вероятность события, бит, аддитивность информации, мера неопределённости
fourier: 86 запросов, например: периодический сигнал, гармоники, дискретный спектр, предельный переход к непериодическому сигналу, спектральная плотность
reformation: 77 запросов, например: кризис авторитета папства, продажа индульгенций, Джон Уиклиф и Ян Гус, христианский гуманизм, книгопечатание
social_contract: 77 запросов, например: естественное право, Гуго Гроций, естественное состояние, легитимность власти

In [7]:
client = WikiClient(contact=CONTACT)
store = CorpusStore(OUT, topics_name="topics_v2.jsonl")
collect(client, store, plans)

[1/8] gradient_descent: уже собрана
[2/8] photosynthesis: уже собрана
[3/8] serf_reform: лучшие по теме ['Отмена крепостного права в России', 'Крепостное право', 'Крепостное право в России'], статей ru 192, en 130; запросов к API 373, 332 с
[4/8] entropy: лучшие по теме ['Энтропия', 'Информационная энтропия', 'Энтропия (значения)'], статей ru 163, en 153; запросов к API 720, 636 с
[5/8] fourier: лучшие по теме ['Преобразование Фурье', 'Быстрое преобразование Фурье', 'Преобразование Лапласа'], статей ru 187, en 171; запросов к API 1108, 960 с
[6/8] reformation: лучшие по теме ['XVI век', 'Реформация', 'Реформация в Речи Посполитой'], статей ru 198, en 182; запросов к API 1527, 1364 с
[7/8] social_contract: лучшие по теме ['Общественный договор', 'Теория справедливости', 'Теория оседлого бандита'], статей ru 194, en 178; запросов к API 1913, 1722 с
[8/8] inflation: лучшие по теме ['Инфляция', 'Галопирующая инфляция', 'Стагфляция'], статей ru 216, en 184; запросов к API 2360, 2077 с


In [8]:
# Что получилось.
articles = store.unique_articles()
by_lang = {lang: [a for a in articles if a.lang == lang] for lang in ("ru", "en")}
for lang, items in by_lang.items():
    sections = sum(len(a.sections) for a in items)
    chars = sum(len(s.text) for a in items for s in a.sections)
    print(f"{lang}: статей {len(items)}, разделов {sections}, символов {chars:,}")
print(f"размер articles.jsonl: {store.articles_path.stat().st_size / 1e6:.1f} МБ")
print()
for record in store.topics.values():
    print(f"{record['id']}: лучшие по теме {record['top_ru']}, ru {len(record['articles_ru'])}, en {len(record['articles_en'])}")

ru: статей 1627, разделов 16904, символов 30,655,856
en: статей 1418, разделов 23106, символов 41,778,193
размер articles.jsonl: 99.1 МБ

gradient_descent: лучшие по теме ['Стохастический градиентный спуск', 'Метод обратного распространения ошибки', 'Онлайновое машинное обучение'], ru 199, en 176
photosynthesis: лучшие по теме ['Фотосинтез', 'Растительные клетки', 'Солнечная энергия'], ru 144, en 137
serf_reform: лучшие по теме ['Отмена крепостного права в России', 'Крепостное право', 'Крепостное право в России'], ru 192, en 130
entropy: лучшие по теме ['Энтропия', 'Информационная энтропия', 'Энтропия (значения)'], ru 163, en 153
fourier: лучшие по теме ['Преобразование Фурье', 'Быстрое преобразование Фурье', 'Преобразование Лапласа'], ru 187, en 171
reformation: лучшие по теме ['XVI век', 'Реформация', 'Реформация в Речи Посполитой'], ru 198, en 182
social_contract: лучшие по теме ['Общественный договор', 'Теория справедливости', 'Теория оседлого бандита'], ru 194, en 178
inflation: л

In [9]:
# Оглавления лучших статей по теме на обоих языках.
for record in store.topics.values():
    for title in record["top_ru"][:2]:
        article = store.articles.get(("ru", title))
        if article:
            heads = [("  " * (s.level - 1)) + (s.heading or "(введение)") + f" [{len(s.text)}]" for s in article.sections]
            print(f"{record['id']} / ru: {article.title}")
            print("\n".join(heads[:20]))
            print()

gradient_descent / ru: Стохастический градиентный спуск
(введение) [885]
Предпосылки [3852]
Итеративный метод [2151]
Пример [9762]
Известные приложения [926]
Разновидности и модификации [938]
  Не выраженные явно изменения (ISGD) [13483]
  Импульс [2034]
  Усреднение [901]
  AdaGrad [4780]
  RMSProp [1605]
  Adam [5678]
  Естественный градиентный спуск и kSGD [7683]
Литература для дальнейшего чтения [1197]

gradient_descent / ru: Метод обратного распространения ошибки
(введение) [1125]
Сигмоидальные функции активации [3479]
Функция оценки работы сети [1685]
Описание алгоритма [29883]
Алгоритм [4928]
  Режимы реализации [1175]
Математическая интерпретация обучения нейронной сети [1236]
Недостатки алгоритма [342]
  Паралич сети [714]
  Локальные минимумы [1068]

photosynthesis / ru: Фотосинтез
(введение) [991]
Классификация [545]
  Бесхлорофилльный фотосинтез [344]
    Открытие у галофильных архей [811]
    Механизм [1357]
    Значение [626]
  Хлорофилльный фотосинтез [308]
    Аноксиген